# Image Tiling

Cuts each image into 256×256 tiles in memory. Nothing is saved to disk.

- `X`: (N, 256, 256, 4) float32. Channels are R, G, B (0-1) and the DEM minus the image mean.
- `y`: (N, 256, 256). 0 = dry, 1 = flood, -1 = unknown.

The split is by whole image, so no tile from a test image ends up in training.

In [ ]:
from pathlib import Path

import numpy as np
import rasterio
import matplotlib.pyplot as plt

## Settings

In [ ]:
DATA_DIR = Path("../data")
TILE = 256

SPLITS = {
    "train": [1, 2, 3, 8, 9],
    "val": [12],
    "test": [4, 5, 6, 7, 10, 11],
}

## Load image

Reads the RGB bands, the alpha band, the DEM and the label.
Rasterio returns bands first `(3, H, W)`, so `transpose` makes it `(H, W, 3)`.
Pixels with alpha 0 have no image data, so their label becomes -1 (unknown).

In [ ]:
def load_image(i):
    folder = DATA_DIR / f"image_{i}"
    with rasterio.open(folder / "image.tif") as src:
        rgb = src.read([1, 2, 3]).transpose(1, 2, 0)
        alpha = src.read(4)
    with rasterio.open(folder / "dem.tif") as src:
        dem = src.read(1)
    with rasterio.open(folder / "label.tif") as src:
        label = src.read(1)

    label[alpha == 0] = -1
    return rgb, dem, label

## Normalize the inputs

- Normalize the RGB values so it's between 0-1
- Subtract the image's mean elevation from the DEM.
- Stack them into one `(H, W, 4)` array.

In [ ]:
def make_input(rgb, dem):
    rgb = rgb / 255
    dem = dem - dem.mean()
    return np.dstack([rgb, dem]).astype(np.float32)

## Cut one image into tiles

In [ ]:
def cut_tiles(x, label, stride, max_unknown):
    H, W = label.shape
    rows = list(range(0, H - TILE, stride)) + [H - TILE]
    cols = list(range(0, W - TILE, stride)) + [W - TILE]

    x_tiles, y_tiles = [], []
    for r in rows:
        for c in cols:
            x_tile = x[r:r + TILE, c:c + TILE]
            y_tile = label[r:r + TILE, c:c + TILE]

            unknown = (y_tile == -1).mean()
            if unknown > max_unknown:
                continue

            x_tiles.append(x_tile)
            y_tiles.append(y_tile)
    return x_tiles, y_tiles

## Tile several images

Runs load → scale → cut for each image and stacks all tiles into two arrays.

In [ ]:
def get_tiles(images, stride=TILE, max_unknown=1.0):
    X, y = [], []
    for i in images:
        rgb, dem, label = load_image(i)
        x = make_input(rgb, dem)
        x_tiles, y_tiles = cut_tiles(x, label, stride, max_unknown)
        X += x_tiles
        y += y_tiles
    return np.stack(X), np.stack(y)

## Make the tiles

Training tiles overlap by half and skip tiles that are mostly unknown. Val and test keep every tile.

In [ ]:
X_train, y_train = get_tiles(SPLITS["train"], stride=128, max_unknown=0.5)
X_val, y_val = get_tiles(SPLITS["val"])
X_test, y_test = get_tiles(SPLITS["test"])

for name, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    known = y[y != -1]
    print(f"{name:5}  {len(X):5} tiles  {X.nbytes / 1e9:.2f} GB  flood {known.mean():.1%}  unknown {(y == -1).mean():.1%}")

## 6. Look at some tiles

Val and test keep every tile, including ones that are almost all unknown, so only tiles at most 50% unknown are shown here.

In [ ]:
def show_tiles(X, y, n=4, seed=None):
    unknown = (y == -1).mean(axis=(1, 2))  # share of unknown pixels per tile
    mostly_known = np.where(unknown <= 0.5)[0]
    picks = np.random.default_rng(seed).choice(mostly_known, n, replace=False)  # seed=None -> new tiles each run

    cmap = plt.get_cmap("Blues").copy()
    cmap.set_bad("red")  # unknown (-1) pixels

    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n), squeeze=False)
    for row, k in enumerate(picks):
        axes[row, 0].imshow(X[k, :, :, :3])
        axes[row, 0].set_title(f"tile {k}: RGB")

        im = axes[row, 1].imshow(X[k, :, :, 3], cmap="terrain")
        axes[row, 1].set_title("DEM (elevation)")
        fig.colorbar(im, ax=axes[row, 1], fraction=0.046)

        axes[row, 2].imshow(np.ma.masked_equal(y[k], -1), cmap=cmap, vmin=0, vmax=1)
        axes[row, 2].set_title("Label (blue = flood, red = unknown)")
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_tiles(X_train, y_train)